In [1]:
!python -V

Python 3.12.1


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
from sklearn.feature_extraction import DictVectorizer

from sklearn.metrics import root_mean_squared_error

In [5]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='./mlruns/1', creation_time=1783510321499, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783510321499, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [6]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    return df

In [7]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [8]:
len(df_train), len(df_val)

(73908, 61921)

In [9]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [10]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [11]:
import xgboost as xgb

In [12]:
#search_space = {
#    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
#    'learning_rate': hp.loguniform('learning_rate', -3, 0),
#    'reg_alpha': hp.loguniform('reg_alpha', -5, 1),
#    'reg_lambda': hp.loguniform('reg_lambda', -6, 1),
#    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
#    'objective': 'reg:linear',
#    'seed': 42
#}

#best_result = fmin(
#    fn=objective,
#    space=search_space,
#    algo=tpe.suggest,
#    max_evals=50,
#    trials=Trials()
#) 

In [15]:
from pathlib import Path

In [16]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [ ]:
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.095853555369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.11658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
            params=best_params,
            dtrain=train,
            num_boost_round=30,
            evals=[(valid, "validation")],
            early_stopping_rounds=50
        )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)  

    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="model")

/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [11:54:10] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44064
[1]	validation-rmse:10.76441
[2]	validation-rmse:10.17574
[3]	validation-rmse:9.66326
[4]	validation-rmse:9.21690
[5]	validation-rmse:8.83366
[6]	validation-rmse:8.50309
[7]	validation-rmse:8.22017
[8]	validation-rmse:7.97540
[9]	validation-rmse:7.76725
[10]	validation-rmse:7.58931
[11]	validation-rmse:7.43705
[12]	validation-rmse:7.30823
[13]	validation-rmse:7.19819
[14]	validation-rmse:7.10465
[15]	validation-rmse:7.02422
[16]	validation-rmse:6.95676
[17]	validation-rmse:6.89749
[18]	validation-rmse:6.84682
[19]	validation-rmse:6.80317
[20]	validation-rmse:6.76623
[21]	validation-rmse:6.73414
[22]	validation-rmse:6.70584
[23]	validation-rmse:6.68127
[24]	validation-rmse:6.65996
[25]	validation-rmse:6.64145
[26]	validation-rmse:6.62664
[27]	validation-rmse:6.61214
[28]	validation-rmse:6.59957
[29]	validation-rmse:6.58865


2026/07/08 11:54:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run judicious-ram-453 at: http://localhost:5000/#/experiments/1/runs/f9f1e21e33724a87ba6e77c19f26f452
🧪 View experiment at: http://localhost:5000/#/experiments/1


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2 Final//EN">
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'